In [5]:
import os
import sys
from pathlib import Path

# Locate the repo root as the directory containing this notebook.
# __vsc_ipynb_file__ is injected by VS Code; fall back to cwd for other environments.
try:
    NOTEBOOK_DIR = str(Path(__vsc_ipynb_file__).parent.resolve())
except NameError:
    NOTEBOOK_DIR = str(Path.cwd())

os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print(f"Working directory: {os.getcwd()}")

Working directory: S:\OneDrive - Umich\Work\Propagation\SWMF-IMF


In [6]:
import pandas as pd
from pyspedas import CDAWeb
from l1_pipeline import get_one_day_swmf_input, create_position_file
from l1_combine import create_combined_l1_files

cda = CDAWeb()

In [7]:

# Force-reload pipeline modules so any code changes made this session
# are picked up without restarting the kernel.
from l1_combine import create_combined_l1_files
from l1_pipeline import get_one_day_swmf_input, create_position_file
import importlib
import l1_pipeline
import l1_combine
import l1_filters
import l1_quality
for mod in [l1_filters, l1_quality, l1_pipeline, l1_combine]:
    importlib.reload(mod)
print('Modules reloaded.')

Modules reloaded.


In [ ]:
# -----------------------------------------------------------------------
# Date range to process.
# Uncomment the rolling-window block to always run the last N days.
# -----------------------------------------------------------------------

# today = pd.Timestamp.utcnow().strftime('%Y-%m-%d')
# days = pd.date_range(end=today, periods=7).strftime('%Y-%m-%d').tolist()

from plot_l1_may2024 import plot_day

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

start_time = pd.Timestamp.now()

# Seed the window: download the day before day-1 (context only) and day-1.
day_before = (pd.Timestamp(days[0]) -
              pd.Timedelta(days=1)).strftime('%Y-%m-%d')
get_one_day_swmf_input(day_before, cda)
get_one_day_swmf_input(days[0], cda)
create_position_file(days[0], cda)

# Crawling window: download tomorrow, combine today (yesterday + tomorrow
# as context), plot, then advance.  Each day is downloaded exactly once.
for i, day in enumerate(days):
    prev_day = day_before if i == 0 else days[i - 1]

    if i < len(days) - 1:
        next_day = days[i + 1]
        get_one_day_swmf_input(next_day, cda)
        create_position_file(next_day, cda)
    else:
        next_day = None

    create_combined_l1_files(day, prev_day=prev_day, next_day=next_day)
    try:
        plot_day(day)
    except Exception as exc:
        print(f"  Plot error for {day}: {exc}")
    print(f"Completed: {day}")

end_time = pd.Timestamp.now()
print(f"\nDone. Elapsed: {end_time - start_time}")

Querying CDAWeb (attempt 1/3)...


23-Mar-26 16:53:00: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240430_v07.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240430_v07.cdf


23-Mar-26 16:53:00: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240430_v07.cdf complete, 0.235 MB in 0.4 sec (0.625 MB/sec) (transfer_normal)
23-Mar-26 16:53:00: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240501_v07.cdf
23-Mar-26 16:53:00: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240501_v07.cdf complete, 0.238 MB in 0.3 sec (0.752 MB/sec) (transfer_normal)
23-Mar-26 16:53:00: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/swepam/level_2_cdaweb/swe_h0/2024/ac_h0_swe_20240430_v11.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\swepam\level_2_cdaweb\swe_h0\2024\ac_h0_swe_20240430_v11.cdf
23-Mar-26 16:53:01: Download of S:\OneDrive - Umich\Work\P


Processing ace...
Saved L1_raw/2024/04/30\L1_ace.dat
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/04/30\L1_ace.dat
Cleaning up ace CDFs...

Processing DSCOVR (NGDC)...
  Downloaded oe_f1m_dscovr_s20240430000000_e20240430235959_p20240501021914_pub.nc.gz -> cdf_temp\dscovr_f1m_20240430.nc.gz
  Downloaded oe_m1m_dscovr_s20240430000000_e20240430235959_p20240501021331_pub.nc.gz -> cdf_temp\dscovr_m1m_20240430.nc.gz
Saved L1_raw/2024/04/30\L1_dscovr.dat
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/04/30\L1_dscovr.dat
  Cleaning up DSCOVR NGDC files...

Processing wind...
Rotating WIND Plasma data from GSE to GSM...
Saved L1_raw/2024/04/30\L1_wind.dat
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/04/30\L1_wind.dat
Cleaning up wind CDFs...
Querying CDAWeb (attempt 1/3)...


23-Mar-26 16:53:16: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240501_v07.cdf


...

23-Mar-26 16:59:34: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240511_v07.cdf complete, 0.243 MB in 0.4 sec (0.594 MB/sec) (transfer_normal)
23-Mar-26 16:59:35: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240512_v07.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240512_v07.cdf
23-Mar-26 16:59:35: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240512_v07.cdf complete, 0.235 MB in 0.4 sec (0.544 MB/sec) (transfer_normal)
23-Mar-26 16:59:35: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/swepam/level_2_cdaweb/swe_h0/2024/ac_h0_swe_20240511_v11.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\swepam\level_2_cdaweb\swe_h0\2024\ac_h0_swe_20240511_v11.cdf
23-Mar-26 16:59:35: Download of S:\OneDrive - Umich\Work\P


Processing ace...
Saved L1_raw/2024/05/11\L1_ace.dat
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/05/11\L1_ace.dat
Cleaning up ace CDFs...

Processing DSCOVR (NGDC)...
  Downloaded oe_f1m_dscovr_s20240511000000_e20240511235959_p20240512022214_pub.nc.gz -> cdf_temp\dscovr_f1m_20240511.nc.gz
  Downloaded oe_m1m_dscovr_s20240511000000_e20240511235959_p20240512021629_pub.nc.gz -> cdf_temp\dscovr_m1m_20240511.nc.gz
Saved L1_raw/2024/05/11\L1_dscovr.dat
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/05/11\L1_dscovr.dat
  Cleaning up DSCOVR NGDC files...

Processing wind...
Rotating WIND Plasma data from GSE to GSM...
Saved L1_raw/2024/05/11\L1_wind.dat
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/05/11\L1_wind.dat
Cleaning up wind CDFs...

--- Generating L1_satpos.dat for 2024-05-11 ---
Querying Position Data (11:00-13:00 UT) (attempt 1/3)...


23-Mar-26 16:59:49: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240511_v07.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240511_v07.cdf


23-Mar-26 16:59:49: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240511_v07.cdf complete, 0.243 MB in 0.4 sec (0.652 MB/sec) (transfer_normal)
23-Mar-26 16:59:49: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/wind/mfi/mfi_h0/2024/wi_h0_mfi_20240511_v05.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\wind\mfi\mfi_h0\2024\wi_h0_mfi_20240511_v05.cdf
23-Mar-26 16:59:49: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\wind\mfi\mfi_h0\2024\wi_h0_mfi_20240511_v05.cdf complete, 2.497 MB in 0.6 sec (4.098 MB/sec) (transfer_normal)
23-Mar-26 16:59:50: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/dscovr/orbit/pre_or/2024/dscovr_orbit_pre_20240511_v02.cdf to S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\dscovr\orbit\pre_or\2024\dscovr_orbit_pre_20240511_v02.cdf
23-Mar-26 16:59:50: Download of S:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\dscovr\orbit\pre_or\2024\

Saved L1/2024/05/11\L1_satpos.dat

Processing L1 data for 2024-05-10  (context window: ['2024-05-09', '2024-05-10', '2024-05-11'])...
  Running plasma quality assessment for all satellites...
  ACE quality: flagged bad: {'Ux': 646, 'Uy': 1405, 'Uz': 1078, 'rho': 1148, 'T': 1698}
  DSCOVR quality: flagged bad: {'Ux': 647, 'Uy': 2685, 'Uz': 2522, 'rho': 1035, 'T': 1698}
  WIND quality: flagged bad: {'Ux': 103, 'Uy': 435, 'Uz': 1225, 'rho': 171, 'T': 3722}
  -> Running Despike (Growth & Change limits)...
Created L1/2024/05/10\L1_combined.dat
  -> Propagating to 14 Re (89194 km)...
    Created L1/2024/05/10\IMF_14Re.dat
  -> Propagating to 32 Re (203872 km)...
    Created L1/2024/05/10\IMF_32Re.dat
Saved plots\2024_05_10.png
Completed: 2024-05-10
Querying CDAWeb (attempt 1/3)...
